# Board Meeting RAG — 03: Summarize Quarter

Produces a neutral factual per-entity summary of the selected quarter's board
meeting materials. Pulls source chunks from
`acme_holdings.documents.board_meetings_chunks` and demonstrates two retrieval
paths:

1. **Direct Delta** — pull every chunk for the quarter (complete, deterministic).
   Used for the summaries so nothing gets missed.
2. **Vector Search** — query-driven retrieval with quarter filter (used at the
   bottom for thematic lookups, e.g., "AI exposure this quarter").

Per-entity summaries are written to
`/Volumes/acme_holdings/documents/board_meetings_raw/_output/summary_<quarter>.md`
and also logged to the `board_meeting_summaries` audit table.


In [ ]:
%pip install --quiet mlflow databricks-vectorsearch
dbutils.library.restartPython()

In [ ]:
from mlflow.deployments import get_deploy_client
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import datetime, json, os

CATALOG = "acme_holdings"
CHUNKS_TABLE = f"{CATALOG}.documents.board_meetings_chunks"
AUDIT_TABLE = f"{CATALOG}.documents.board_meeting_summaries"
INDEX_NAME = f"{CATALOG}.embeddings.board_meetings_index"
ENDPOINT_NAME = "acme_rag_endpoint"
OUTPUT_VOLUME_DIR = f"/Volumes/{CATALOG}/documents/board_meetings_raw/_output"

dbutils.widgets.text("quarter", "2026Q1")
QUARTER = dbutils.widgets.get("quarter")

print(f"Generating per-entity summaries for quarter: {QUARTER}")

SUMMARY_MODEL = "databricks-claude-sonnet-4-6"

client = get_deploy_client("databricks")

## Pull all chunks for the quarter, grouped by entity

In [ ]:
chunks_df = (
    spark.table(CHUNKS_TABLE)
    .filter(F.col("quarter") == QUARTER)
    .orderBy("entity", "meeting_date", "page_number")
)

chunks = [r.asDict() for r in chunks_df.collect()]
assert chunks, f"No chunks found for quarter={QUARTER}. Run the ingestion notebook first."

by_entity: dict[str, list[dict]] = {}
for c in chunks:
    by_entity.setdefault(c["entity"], []).append(c)

print(f"Found {len(chunks)} chunks across entities: {sorted(by_entity)}")

## Per-entity factual summary (Claude Sonnet 4.6)

In [ ]:
def chat(model: str, system: str, user: str, max_tokens: int = 1200, temperature: float = 0.2) -> str:
    resp = client.predict(
        endpoint=model,
        inputs={
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            "max_tokens": max_tokens,
            "temperature": temperature,
        },
    )
    return resp["choices"][0]["message"]["content"].strip()

ENTITY_SUMMARY_SYSTEM = (
    "You are summarizing institutional investment board meeting materials. "
    "Write in a neutral, factual tone. Do not editorialize, speculate, or add "
    "market commentary beyond what is stated in the source. Preserve specific "
    "numbers and named entities exactly as written."
)

def build_entity_summary(entity: str, rows: list[dict]) -> str:
    body = "\n\n".join(
        f"[{r['source_file']} | page {r['page_number']} | section {r['section']}]\n{r['content']}"
        for r in rows
    )
    user = (
        f"Summarize the {entity} board meeting materials for {QUARTER} below in 4-7 bullet "
        f"points. Focus on: performance vs benchmark, portfolio / allocation changes, new "
        f"commitments, liquidity, and any policy decisions. Keep numbers exact.\n\n"
        f"=== SOURCE MATERIAL ===\n{body}"
    )
    return chat(SUMMARY_MODEL, ENTITY_SUMMARY_SYSTEM, user, max_tokens=900)

entity_summaries = {e: build_entity_summary(e, rows) for e, rows in by_entity.items()}
for e, s in entity_summaries.items():
    print(f"===== {e} =====\n{s}\n")

## Persist: write summaries to Volume + log to audit table

In [ ]:
os.makedirs(OUTPUT_VOLUME_DIR, exist_ok=True)
out_path = f"{OUTPUT_VOLUME_DIR}/summary_{QUARTER}.md"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(f"# Board meeting summaries — {QUARTER}\n\n")
    f.write(f"_Generated: {datetime.datetime.utcnow().isoformat()}Z_\n\n")
    for e, s in entity_summaries.items():
        f.write(f"## {e}\n\n{s}\n\n")
print(f"Wrote {out_path}")

In [ ]:
# The audit table carries linkedin_post/model_post columns for downstream
# distribution steps that aren't part of this pipeline. We write NULL for those
# fields here and leave the schema intact for any later step that wants to populate them.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
  quarter STRING,
  generated_at TIMESTAMP,
  entity_summaries STRING,
  linkedin_post STRING,
  model_post STRING,
  model_summary STRING
) USING DELTA
""")

audit_schema = StructType([
    StructField("quarter", StringType(), False),
    StructField("generated_at", TimestampType(), False),
    StructField("entity_summaries", StringType(), True),
    StructField("linkedin_post", StringType(), True),
    StructField("model_post", StringType(), True),
    StructField("model_summary", StringType(), True),
])

row = Row(
    quarter=QUARTER,
    generated_at=datetime.datetime.utcnow(),
    entity_summaries=json.dumps(entity_summaries),
    linkedin_post=None,
    model_post=None,
    model_summary=SUMMARY_MODEL,
)
spark.createDataFrame([row], schema=audit_schema).write.mode("append").saveAsTable(AUDIT_TABLE)

## Bonus: thematic Vector Search retrieval (works across quarters)

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)
idx = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)

for query in [
    "AI exposure and venture capital themes",
    "liquidity and net benefit payments",
    "new private equity commitments",
]:
    print(f"\n=== {query} ===")
    res = idx.similarity_search(
        query_text=query,
        columns=["entity", "quarter", "section", "page_number", "content"],
        filters={"quarter": QUARTER},
        num_results=2,
    )
    for row in res.get("result", {}).get("data_array", []):
        print(f"[{row[0]} {row[1]} p{row[3]}] {(row[4] or '')[:240]}…")